# WarpTracer PPO
Select **Runtime → Change runtime type → GPU**. Setup pulls `performance-prototype` and installs the optional Torch training dependencies. Start with `oval` (1,024 independent cars); `large` uses 4,096. Each cell streams output.


In [ ]:
import os, sys, subprocess, shutil, signal
from pathlib import Path

def run_stream(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
        raise

run_stream(["nvidia-smi"])
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    run_stream(["git", "clone", "--branch", "performance-prototype",
                "https://github.com/yuywe/WarpTracer.git", str(repo)])
os.chdir(repo)
run_stream(["git", "fetch", "origin", "performance-prototype"])
run_stream(["git", "switch", "performance-prototype"])
run_stream(["git", "pull", "--ff-only"])
os.environ["UV_PYTHON"] = sys.executable
run_stream(["uv", "sync", "--locked", "--extra", "rl"])
run_stream(["uv", "run", "--locked", "--extra", "rl", "python", "-c",
            "import torch, warp as wp; wp.init(); assert torch.cuda.is_available() and wp.is_cuda_available(); print(torch.cuda.get_device_name(0))"])


## Train
The default runs 300 PPO updates and saves `outputs/ppo/checkpoint.pt`. Use `resume = True` to add another 300 updates from the last saved checkpoint. A recovery checkpoint is saved before training, even if the first update is interrupted. Resuming trims logs to that saved update.


In [ ]:
preset = "oval"  # "large" for 4096 cars; "smoke" for a tiny CPU check
resume = False
command = ["uv", "run", "--locked", "--extra", "rl", "warptracer", "train", preset]
if resume:
    command += ["--resume", "outputs/ppo/checkpoint.pt"]
run_stream(command)


## Inspect learning
Reward alone is insufficient: check completed episodes and successful laps. The two throughput values include PPO inference and, for training, optimization.


In [ ]:
import json
import matplotlib.pyplot as plt
history = [json.loads(line) for line in Path("outputs/ppo/training.jsonl").read_text().splitlines()]
print("Last update:", history[-1])
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
x = [r["transitions"] for r in history]
axes[0].plot(x, [r["mean_step_reward"] for r in history]); axes[0].set_title("Mean step reward")
axes[1].plot(x, [r["successful_laps"] for r in history]); axes[1].set_title("Completed laps per rollout")
axes[2].plot(x, [r["training_transitions_per_second"] for r in history]); axes[2].set_title("Training transitions/s")
for ax in axes: ax.set_xlabel("Aggregate environment transitions")
plt.tight_layout()


## Evaluate and download
This records one deterministic episode in a separate environment, including its initial and terminal car/LiDAR state. Download the zip, then open `evaluation.html` in your browser to orbit, pause and scrub the interactive replay. Training itself has no live 3D viewer.


In [ ]:
run_stream(["uv", "run", "--locked", "--extra", "rl", "warptracer", "evaluate"])
from google.colab import files
archive = shutil.make_archive("/content/warptracer-ppo", "zip", root_dir="outputs/ppo")
files.download(archive)
